# Python Crime Data Engineering Pipeline Project

### Introduction 


project introduction

### Workflow Diagram 

*insert data diagram

### Data Assumptions

• Data assumption 1...  
• Data assumption 2...

### Import Python Libraries

In [2]:
#data handling libraries 
import pandas as pd 
import numpy as np

#file and path management 
from pathlib import Path 
import os 
import glob

### Ingestion Layer

**Outline:** 

This layer follows the steps taken to load data into the notebook for later cleaning and transformation. 

**1a. Load Crime Data in batches, by police force**

The data is loaded in batches to avoid loading the full dataset into memory at once. This ensures a more scaleable pipeline and an easier debugging by police force process.  

For example, when an error occurs, Python can specify which batch(police force) caused this error, narrowing down the number of files to check. 

**1b. Validate file upload**

The number of rows and files loaded will be printed. This allows for loading validation and comparison between raw and processed data. 

In [11]:
#define path to raw crime data folder 
raw_data = Path("data")/ "raw_data" / "crime_data" #create path object 

#create a list of the forces to loop through
forces = ['metropolitan','thames-valley','west-midlands','west-yorkshire']

#create an empty list for each batch of forces to be added to 
batches = [] 

#create a for loop that locates individual police force files and adds them to the batches list 
for force in forces: 
    #create a list of csv files to loop through 
    csv_files = list(raw_data.glob(f"**/*{force}*street.csv")) #** - glob searches in sub-folders, * - wildcard spot 

    #create an empty list for each file per force to be added to, resets every outer loop iteration  
    force_batch = []
    #create a nested for loop that reads each police force file in batches 
    for file in csv_files: 
        df = pd.read_csv(file) #create a dataframe for each file 
        force_batch.append(df) #add each dataframe to force_batch list 

    #create a dataframe with all files from each force 
    force_df = pd.concat(force_batch, ignore_index=True) #ignore_index=True - resets index + prevents duplicates in force_df 
    print(f"\n{force}: {len(force_df)} rows loaded from {len(csv_files)} files\n") #validation check 
    batches.append(force_df) #add each force batch to batches list 

#once all files have been retrieved from each force, combine all files into one raw dataframe 
crime_raw = pd.concat(batches, ignore_index=True)
print(f"\nTotal rows loaded: {len(crime_raw)}\n")


metropolitan: 2386012 rows loaded from 25 files


thames-valley: 413128 rows loaded from 25 files


west-midlands: 712238 rows loaded from 25 files


west-yorkshire: 647272 rows loaded from 25 files


Total rows loaded: 4158650



**2. Load Enrichment Data**

The load_file function is created as a standardised step to reduce repetition in the code. For example, the file type check and print statement only need to be written once inside the function. When calling the function, arguments will be specified individually to account for the varying file structures. 

In [3]:
# create a function to load excel/csv files

def load_file(filepath, sheet_name=None, header=0): 
    """ 
    loads a CSV or Excel file based on the file extension. 
    
    Parameters: 
        filepath = path to the file (defined when the function is called)  
        sheet_name = Excel sheet name (only for Excel files)
        header = row to use as column headers - 0 indexed (only for Excel files)
        
    Returns: 
    df = dataframe to assign to a variable 
    """
    filepath = Path(filepath) #converts first argument to a path object 

    if filepath.suffix == '.csv': #checks file suffix (pathlib attribute)
        df = pd.read_csv(filepath)
    elif filepath.suffix in ['.xlsx','.xls']: #accounts for multiple Excel file suffix'
        df = pd.read_excel(filepath, 
                          sheet_name = sheet_name, 
                           header = header 
                          ) #assigns what ever is passed when the function is called 
    else: 
        raise ValueError(f"Unsupported file type: {filepath.suffix}") #clearly define error 

    print (f"{filepath.name}: {len(df)} rows loaded.") #prints full file name + suffix (pathlib attribute)
    return df 
        
#define path to raw data folder 
raw_data = Path("data")/ "raw_data"

**2a. Load Population enrichment data**

In [7]:
pop_raw = load_file(raw_data/ "policeforceareas1991to2024.xlsx",
                        sheet_name="Mid-2021 to Mid-2024", 
                        header=3) #assigns 4th row (0 indexed)as column headers 

policeforceareas1991to2024.xlsx: 172 rows loaded.


**2b. Load Deprivation enrichment data**

In [8]:
dep_raw = load_file(raw_data/"deprivation_data.csv")

deprivation_data.csv: 33755 rows loaded.


**2c. Load Income & Housing enrichment data**

In [9]:
#load median house price sheet 
hos_raw = load_file(raw_data/ "aff1ratioofhousepricetoworkplacebasedearnings.xlsx",
                        sheet_name="5a", 
                        header=1) #assigns 2nd row (0 indexed)as column headers 

#load median workplace-based earnings sheet 
inc_raw = load_file(raw_data/ "aff1ratioofhousepricetoworkplacebasedearnings.xlsx",
                        sheet_name="5b", 
                        header=1) #assigns 2nd row (0 indexed)as column headers 

aff1ratioofhousepricetoworkplacebasedearnings.xlsx: 318 rows loaded.
aff1ratioofhousepricetoworkplacebasedearnings.xlsx: 318 rows loaded.


### Cleaning & Validation Layer


**Outline:** 

This layer follows the steps taken to clean the data, including data quality assessments... 

**1. Data Quality Assessment**

The qual_check function is created as a standardised step to observe initial dataset impressions including column names, row & column count, datatypes, nulls, etc., reducing repitition in the code. When calling the function, arguments will be specified individually to account for the varying dataset structures. 

In [43]:
#create a function to check for data quality issues 

def qual_check(df, name, cat_cols=None):
    """ 
    perfoms a data quality check on a given dataframe. 

    Parameters: 
    df = given dataframe to assess
    name = variable name given to the dataframe (used in print statements)
    cat_cols = list of categorical columns to check for format incosistencies/unique values  
    """
    print(f"Data Quality Check: {name}")

    #dataset preview 
    print("\nFirst 3 Rows:\n")
    display(df.head(3))

    #dataset shape 
    print(f"\nNumber of Rows and Columns: {df.shape}")

    #column data types 
    print(f"\nColumn Data Types:\n\n{df.dtypes}")

    #null summary 
    null_counts = df.isnull().sum()
    null_pct = (df.isnull().sum() / len(df) * 100).round(0)
    null_summary = pd.DataFrame({ #create null summary dataframe 
    'Null Count': null_counts, 
    'Null %': null_pct
    })
    print("\nColumns with null values:\n")
    display(null_summary[null_summary['Null Count'] > 0]) #only shows columns with null values

    #duplicate check 
    print(f"\nRow Duplicates Count: {df.duplicated().sum()}")

    #categorical columns unique values check 
    if cat_cols: 
        print("\nCategorical Columns Unique Values:\n")
        for col in cat_cols: 
            if col in df.columns: 
                print(f"\nColumn Name: {col}\n")
                print(f"\nTotal unique values: {df[col].nunique()}\n")
                print(f"\nUnique values:\n{df[col].unique()}") 
            else: 
                raise ValueError(f"Column '{col}' not found in '{name}'"
                                f"Available columns are: {df.columns.tolist()}")           

**1a. Crime Data**

Data Quality Issues: 

***Null values***  
    • Context = 100% null  
    • Crime ID = 16% null  
    • Last outcome category = 16% null  
    • Longitude = 1% null  
    • Latitude = 1% null  
    • LSOA code = 1% null  
    • LSOA name = 1% null   

Where LSOA code is null, LSOA name, Longitude, and Latitude is also null. The null LSOA code and LSOA name values limits merge ability with dep_raw, resulting in a loss of deprivation data. See step x for data cleaning process.

***Multi-Value attributes***     
    • Month - stores month and year info   

This multi-value attribute will impact the final reporting grain. See step x for data cleaning process. 

***Duplicate Records***      
    • 262,672 duplicate records in the dataset  

Duplicate records may skew analysis. See step x for data cleaning process.
    

In [ ]:
qual_check(crime_raw, "Raw Crime Data", cat_cols=['Reported by', 'Falls within', 'LSOA name', 'Crime type', 'Last outcome category'])

In [ ]:
display(crime_raw[crime_raw['LSOA code'].isnull()])

**1b. Population Data**

Data Quality Issues: 

***Inconsistent Format*** of police force names between pop_raw & crime_raw  

• Metropolitan Police vs Metropolitan Police Service  
• Thames Valley vs Thames Valley Police  
• West Midlands vs West Midlands Police     
• West Yorkshire vs West Yorkshire Police  

These inconsistency may cause issues when merging the pop_raw and crime_raw datasets. See step x for data cleaning process.

In [ ]:
qual_check(pop_raw, "Raw Population Data", cat_cols=['PFA 2023 Name'])

**1c. Deprivation Data**

Data Quality Issues: 

***Long Column Names***

May reduce readability of final dataset and dashboard. See step x for data cleaning process.

In [ ]:
qual_check(dep_raw, "Raw Deprivation Data", cat_cols=['LSOA name (2021)', 'Local Authority District name (2024)'])

**1d. Housing Data**

Data Quality Issues: 

***Incompatible Grains***

Each record in the dataset has a median house price for each year (each year has it's own column). Unlike the crime_raw dataset, where each record refers to a specified month/year, only storing data relevant to that month/year. This may cause issues when merging the datasets. See step x for the data cleaning process. 

In [ ]:
qual_check(hos_raw, "Raw Housing Data", cat_cols=['Country/Region name', 'Local authority name'])

**1e. Income Data**

Data Quality Issues: 

***Incompatible Grains***

Each record in the dataset has a median income for each year (each year has it's own column). Unlike the crime_raw dataset, where each record refers to a specified month/year, only storing data relevant to that month/year. This may cause issues when merging the datasets. See step x for the data cleaning process. 

***Income data stored as object***

This may slow performance or cause errors later in analysis. See step x for the data cleaning process.

In [ ]:
qual_check(inc_raw, "Raw Income Data", cat_cols=['Country/Region name', 'Local authority name'])

**2. Clean Data**

The "" function is created as a standardised step to clean the datasets based on the discovered data quality issues. When calling the function, the arguments...

Data Cleaning Steps: 

1. Create a copy of the raw data to allow access and comparison to the original data
2. Remove duplicates rows before other data cleaning steps to reduce processing time by avoiding unnecassary cleaning of rows 
3. Drop columns with more than 50% null values before other data cleaning steps to reduce processing time by avoiding filling in null values that will be dropped
4. Fill null values according to data type to prevent later aggregation errors 

In [ ]:
#create a function that corrects the data quality issues found

def clean_data(df): 
    """
    performs data cleaning tasks on a given dataframe. 

    Parameters: 
    df = given dataframe to clean

    Returns:
    df = cleaned dataframe 
    """
    clean_df = df.copy() 

    #remove duplicate rows 
    dupes = clean_df.duplicated().sum()
    if dupes > 0: 
        print(f"Dropping {dupes} duplicate rows\n")
        clean_df = clean_df.drop_duplicates().reset_index(drop=True)
        print (f"Number of rows after dropping duplicates: {len(clean_df)}\n")
    
    #drop columns with > 50% nulls 
    null_pct = (clean_df.isnull().sum() / len(clean_df) * 100).round(0) #creates a pandas series (list of dataframe columns)
    null_cols = null_pct[null_pct > 50].index.tolist() #returns row lables of columns with > 50% nulls and adds them to a list 

    if col in null_cols: 
        print(f"\nDropping columns: {null_cols}\n")
        clean_df = clean_df.drop(columns=null_cols)  
        print(f"Number of columns after dropping nulls: {clean_df.shape[1]\n")

    #fill other null values with 0, 0.0 or Unknown"
    for col in clean_df.columns: #loops through a list of dataframe columns 
        
        null_count = clean_df[col].isnull().sum()
        print(f"\nFilling {null_count} null values in '{col}'")
        
        if pd.api.types.is_integer_dtype(clean_df[col]): #checks if dataframe column values are integers
            clean_df[col] = clean_df[col].fillna(0) #fills nulls with 0 in clean_df 

        elif pd.api.types.is_float_dtype(clean_df[col]): #checks if dataframe column values are floats
           clean_df[col] = clean_df[col].fillna(0.0) #fills nulls with 0.0 in clean_df

        else: #treats dataframe column values as objects/strings if they are not numerical 
            clean_df[col] = clean_df[col].fillna("Unknown") #fills nulls with "Unknown" in clean_df  
            
        print(f"\nNumber of null values filled: {null_count}")

    #split month column into year and month 
    if 'Month' in clean_df.columns: 
        print("Splitting 'Month' column into 'Year' and 'Month'")
        dates = pd.to_datetime(clean_df['Month'], format="%Y-%m") #creates a series (list of column values)

        month_index = clean_df.columns.get_loc("Month") #finds location of Month column 

        clean_df.insert(month_index + 1, "Year", dates.dt.year) #inserts a new Year column, defining it's position, name, and values (extracted years) 
        clean_df['Month'] = dates.dt.month #overwrites month column with extracted months 
        print("'Month' column split into 'Year' and 'Month'")
   

**3a. Crime Data**

Data cleaning steps performed: 


**3. Data validation check**

The number of rows will be printed after cleaning and compared with raw data row count. There will also be a duplicate check at the intended reporting grain. 

### Feature Engineering & Transformation Layer

**Outline:** 

• 

### Aggregation for Reporting


**Outline:** 

• 

In [ ]:
#aggregate final outcome by mode 

### Export & Final Validation


**Outline:** 

• 

In [ ]:
# Grain level - check for duplicates at reporting grain before aggregation
print(f"Duplicate rows at reporting grain: {crime_raw.duplicated(subset=['Falls within', 'Month', 'Crime type']).sum()}")

### Data Dictionary  

*insert data dictionary 

### Conclusion 

brief conclusion